In [1]:
!pip install pyspark==3.5.0 delta-spark==3.1.0

## Step 1 : Import Required Libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

## Step 2 : Create Spark Session with Delta Support

In [3]:
builder = SparkSession.builder \
    .appName("DeltaMerge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


## Step 3 : Upload Dataset

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore (1).csv


## Step 4 : Load CSV Dataset

In [5]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Step 5 : Explore Dataset

In [6]:
print("Number of Rows :", df.count())
print("Number of Columns :", len(df.columns))

df.printSchema()

Number of Rows : 9994
Number of Columns : 21
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Step 6 : Check Missing Values

In [7]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [8]:
print("No missing values found in the dataset.")
print("Data cleaning for missing values is not required.")

No missing values found in the dataset.
Data cleaning for missing values is not required.


## Step 8: Check for Duplicate Records

In [9]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total Rows :", total_rows)
print("Unique Rows :", unique_rows)

if total_rows == unique_rows:
    print("No duplicate records found.")
else:
    print("Duplicate records found :", total_rows - unique_rows)

Total Rows : 9994
Unique Rows : 9994
No duplicate records found.


In [10]:
print("No duplicate records found in the dataset.")
print("Duplicate removal is not required.")

No duplicate records found in the dataset.
Duplicate removal is not required.


## Step 10: Rename Column Names

Delta Lake does not support spaces and certain special characters in column names. Therefore, the column names are standardized by replacing spaces and hyphens with underscores before saving the dataset as a Delta table.

In [11]:
new_columns = []

for col_name in df.columns:
    col_name = col_name.replace(" ", "_")
    col_name = col_name.replace("-", "_")
    new_columns.append(col_name)

df = df.toDF(*new_columns)


print(df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Step 11: Save the Dataset as a Delta Table

In [12]:
delta_path = "/content/delta_superstore"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Dataset successfully saved as a Delta Table.")

Dataset successfully saved as a Delta Table.


## Step 12: Load the Delta Table

In [13]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

print("Delta Table Loaded Successfully.")

Delta Table Loaded Successfully.


In [14]:
delta_df = spark.read.format("delta").load(delta_path)

delta_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Step 13: Create an Incremental Dataset

In [15]:
incremental_data = [

# Existing Record (Update)
(
    1,
    "CA-2016-152156",
    "11/8/2016",
    "11/11/2016",
    "Second Class",
    "CG-12520",
    "Claire Gute",
    "Consumer",
    "United States",
    "Henderson",
    "Kentucky",
    42420,
    "South",
    "FUR-BO-10001798",
    "Furniture",
    "Bookcases",
    "Bush Somerset Collection Bookcase",
    300.0,
    2,
    0.0,
    90.0
),

# New Record (Insert)
(
    99999,
    "NEW-100001",
    "07/05/2026",
    "07/07/2026",
    "Standard Class",
    "AB-10001",
    "John Smith",
    "Consumer",
    "United States",
    "Austin",
    "Texas",
    73301,
    "Central",
    "TEC-PH-1000001",
    "Technology",
    "Phones",
    "Apple iPhone 15",
    1200.0,
    1,
    0.0,
    300.0
)

]

## Step 14: Convert Incremental Data into a DataFrame

In [16]:
incremental_df = spark.createDataFrame(
    incremental_data,
    schema=df.schema
)

incremental_df.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+---------+--------+-----------+-------+---------------+----------+------------+---------------------------------+------+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment |Country      |City     |State   |Postal_Code|Region |Product_ID     |Category  |Sub_Category|Product_Name                     |Sales |Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+--------+-------------+---------+--------+-----------+-------+---------------+----------+------------+---------------------------------+------+--------+--------+------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  |Consumer|United States|Henderson|Kentucky|42420      |South  |FUR-BO-10001798|Furniture |Bookcases   |Bush Somerset Collection Bookcase|300.0 |2     

## Step 15: Save the Incremental Dataset

In [17]:
incremental_df.toPandas().to_csv(
    "/content/superstore_incremental.csv",
    index=False
)

print("Incremental dataset saved successfully.")

Incremental dataset saved successfully.


In [18]:
from google.colab import files

files.download("/content/superstore_incremental.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 16: Perform MERGE Operation

In [19]:
delta_table.alias("target").merge(
    incremental_df.alias("source"),
    """
    target.Order_ID = source.Order_ID
    AND target.Product_ID = source.Product_ID
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("MERGE operation completed successfully.")

MERGE operation completed successfully.


## Step 17: Read the Updated Delta Table

In [20]:
final_df = spark.read.format("delta").load(delta_path)

final_df.show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+-----------------------------------------------------+------------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name   |Segment    |Country      |City         |State     |Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                                         |Sales       |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+-----------------------------------------------------+------------+--------+--------+--------+
|6288  |CA-2014-100090|7/8/2014  |7/12/2014 |Standard Class|EB-13705   |Ed Braxton      |Corporate  

## Step 18: Validate the MERGE Operation

In [21]:
print("Original Records :", df.count())
print("Final Records :", final_df.count())

Original Records : 9994
Final Records : 9995


## Step 19: Verify the Updated Record

In [22]:
final_df.filter(
    (col("Order_ID") == "CA-2016-152156") &
    (col("Product_ID") == "FUR-BO-10001798")
).show(truncate=False)

+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+-----+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode   |Customer_ID|Customer_Name|Segment |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category |Sub_Category|Product_Name                     |Sales|Quantity|Discount|Profit|
+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+-----+--------+--------+------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class|CG-12520   |Claire Gute  |Consumer|United States|Henderson|Kentucky|42420      |South |FUR-BO-10001798|Furniture|Bookcases   |Bush Somerset Collection Bookcase|300.0|2       |0.0     |90.0  |


## Step 20: Verify the Newly Inserted Record


In [23]:
final_df.filter(
    col("Order_ID") == "NEW-100001"
).show(truncate=False)

+------+----------+----------+----------+--------------+-----------+-------------+--------+-------------+------+-----+-----------+-------+--------------+----------+------------+---------------+------+--------+--------+------+
|Row_ID|Order_ID  |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment |Country      |City  |State|Postal_Code|Region |Product_ID    |Category  |Sub_Category|Product_Name   |Sales |Quantity|Discount|Profit|
+------+----------+----------+----------+--------------+-----------+-------------+--------+-------------+------+-----+-----------+-------+--------------+----------+------------+---------------+------+--------+--------+------+
|99999 |NEW-100001|07/05/2026|07/07/2026|Standard Class|AB-10001   |John Smith   |Consumer|United States|Austin|Texas|73301      |Central|TEC-PH-1000001|Technology|Phones      |Apple iPhone 15|1200.0|1       |0.0     |300.0 |
+------+----------+----------+----------+--------------+-----------+-------------+--------+-----

In [24]:
final_df.groupBy("Order_ID", "Product_ID") \
        .count() \
        .filter("count > 1") \
        .show()

+--------------+---------------+-----+
|      Order_ID|     Product_ID|count|
+--------------+---------------+-----+
|US-2016-123750|TEC-AC-10004659|    2|
|CA-2016-140571|OFF-PA-10001954|    2|
|CA-2017-118017|TEC-AC-10002006|    2|
|US-2014-150119|FUR-CH-10002965|    2|
|CA-2016-137043|FUR-FU-10003664|    2|
|CA-2016-129714|OFF-PA-10001970|    2|
|CA-2017-152912|OFF-ST-10003208|    2|
|CA-2015-103135|OFF-BI-10000069|    2|
+--------------+---------------+-----+



## Step 21: Display Final Dataset


In [25]:
final_df.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------------------------+------------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name    |Segment    |Country      |City         |State     |Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                                                                         |Sales       |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+-------------------------------------------------------------------------------------+------------+--------+--------+--------+
|